# Lab 07: Latent Spaces

Uses `lab07_starter.py` unchanged. 8 by 8 digits, 1,438 training and 359 test images, pixels in [0, 1]. Every model has an 8-dimensional code and one hidden layer of 64 units on each side, and trains for 200 epochs at 2e-3 with batch 64 on seeds 0, 1 and 2 unless noted.

In [1]:
import json
import os
import subprocess
import sys
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import lab07_starter as S

SEEDS = [0, 1, 2]
LATENT_DIM = 8
HIDDEN_DIM = 64
EPOCHS, LEARNING_RATE, BATCH_SIZE = 200, 2e-3, 64
N_SAMPLES = 359  # every Frechet distance in this notebook uses 359 images
os.makedirs("figures", exist_ok=True)
results = {}
notebook_start = time.perf_counter()

## Run the starter once

Before writing anything I want to know the environment is fine, so I run the starter file exactly as it was handed out. A pass is the script finishing and printing a reconstruction error and a set of sample scores.

In [2]:
starter_start = time.perf_counter()
starter_run = subprocess.run([sys.executable, "lab07_starter.py"], capture_output=True, text=True, check=True)
print(starter_run.stdout)
print(f"starter took {time.perf_counter() - starter_start:.1f}s")

train (1438, 64) test (359, 64)
autoencoder test reconstruction MSE: 0.0163
feature network accuracy on real test images: 0.972
autoencoder samples from N(0, I): {'frechet': 86.242, 'precision': 0.437, 'recall': 0.794, 'nearest_train': 2.033}
saved starter_check.png

starter took 9.1s


It finished in well under a minute. The feature network gets 0.972 on the real test images, which is what every sample score below leans on.

## Load the data and train the one feature network

Every Fréchet distance, precision and recall in this lab has to come from the same feature network, otherwise the numbers can't be compared. I train it once here with the starter's defaults (seed 0) and reuse `feature_net` and `test_features` for every score.

In [3]:
Xtr, ytr, Xte, yte = S.load_task()
print("train", tuple(Xtr.shape), "test", tuple(Xte.shape))
print(f"pixel range: {Xtr.min().item():.2f} to {Xtr.max().item():.2f}")
print("train images per digit:", torch.bincount(ytr, minlength=10).tolist())
print("test images per digit: ", torch.bincount(yte, minlength=10).tolist())

feature_net = S.train_feature_net(Xtr, ytr)
test_features = S.get_features(feature_net, Xte)
with torch.no_grad():
    feature_net_accuracy = (feature_net(Xte).argmax(1) == yte).float().mean().item()
print(f"feature network test accuracy: {feature_net_accuracy:.3f}   features per image: {test_features.shape[1]}")
results["feature_net_accuracy"] = feature_net_accuracy

train (1438, 64) test (359, 64)
pixel range: 0.00 to 1.00
train images per digit: [151, 147, 141, 154, 151, 142, 137, 140, 136, 139]
test images per digit:  [27, 35, 36, 29, 30, 40, 44, 39, 38, 41]


feature network test accuracy: 0.972   features per image: 32


Same 0.972 as the starter's own run, so `train_feature_net` is deterministic and the network I score with is the one the starter checked.

## Write the VAE, the KL term and the loss

The starter has no VAE code, so this is all mine. The encoder ends in two heads, a mean and a log-variance, the code is drawn as `mean + std * noise` so the noise is the only random part, and the decoder returns logits for a Bernoulli likelihood on each pixel.

**Convention:** the reconstruction term is summed over the 64 pixels and the KL is summed over the 8 code dimensions, which gives one number per image. The loss is the mean of `reconstruction + beta * KL` over the images in the batch. I keep this in every experiment.

In [4]:
class VAE(nn.Module):
    def __init__(self, d_in=64, d_hidden=HIDDEN_DIM, d_latent=LATENT_DIM):
        super().__init__()
        self.d_latent = d_latent
        self.enc_body = nn.Sequential(nn.Linear(d_in, d_hidden), nn.ReLU())
        self.enc_mean = nn.Linear(d_hidden, d_latent)
        self.enc_logvar = nn.Linear(d_hidden, d_latent)
        self.dec = nn.Sequential(nn.Linear(d_latent, d_hidden), nn.ReLU(), nn.Linear(d_hidden, d_in))

    def encode(self, x):
        hidden = self.enc_body(x)
        return self.enc_mean(hidden), self.enc_logvar(hidden)

    def reparameterize(self, mean, logvar):
        noise = torch.randn_like(mean)               # the only random part, so gradients flow through mean and std
        return mean + torch.exp(0.5 * logvar) * noise

    def decode(self, z):
        return self.dec(z)                           # logits, one per pixel

    def forward(self, x):
        mean, logvar = self.encode(x)
        z = self.reparameterize(mean, logvar)
        return self.decode(z), mean, logvar

    @torch.no_grad()
    def sample(self, n, generator=None):
        z = torch.randn(n, self.d_latent, generator=generator)
        return torch.sigmoid(self.decode(z))         # pixel probabilities, in [0, 1]


def kl_to_standard_normal(mean, logvar):
    """Closed-form KL(N(mean, exp(logvar)) || N(0, I)), summed over code dimensions. One value per image."""
    return 0.5 * (mean ** 2 + torch.exp(logvar) - 1.0 - logvar).sum(dim=1)


def reconstruction_term(logits, x):
    """Negative Bernoulli log-likelihood of the pixels, summed over the 64 pixels. One value per image."""
    return F.binary_cross_entropy_with_logits(logits, x, reduction="none").sum(dim=1)


def vae_terms(model, xb):
    logits, mean, logvar = model(xb)
    return reconstruction_term(logits, xb), kl_to_standard_normal(mean, logvar)


def make_vae_loss(beta):
    def vae_loss(model, xb, yb):
        recon, kl = vae_terms(model, xb)
        return (recon + beta * kl).mean()            # mean over the batch
    return vae_loss


torch.manual_seed(0)
vae = VAE()
print(vae)
print("parameters:", sum(p.numel() for p in vae.parameters()))

VAE(
  (enc_body): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
  )
  (enc_mean): Linear(in_features=64, out_features=8, bias=True)
  (enc_logvar): Linear(in_features=64, out_features=8, bias=True)
  (dec): Sequential(
    (0): Linear(in_features=8, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
  )
)
parameters: 9936


9,936 parameters. Same 64-unit hidden layer as the starter's `AE` on each side, the only extra is the second 8-unit head for the log-variance.

## Check 1: closed-form KL against a Monte Carlo estimate

If the closed form is right it has to agree with the brute-force version, the average of `log q(z) - log p(z)` over many draws from `q`. I use random means and log-variances (spread wide on purpose) and 200,000 draws for each row. A pass is every row within a few standard errors. I also print the closed form averaged over the 8 dimensions instead of summed, which is the mistake the handout warns about, so I can see how far off it would be.

In [5]:
torch.manual_seed(0)
check_means = 1.5 * torch.randn(6, LATENT_DIM)
check_logvars = torch.randn(6, LATENT_DIM)
closed_form_kl = kl_to_standard_normal(check_means, check_logvars)

posterior = torch.distributions.Normal(check_means, torch.exp(0.5 * check_logvars))
prior = torch.distributions.Normal(0.0, 1.0)
draws = posterior.sample((200_000,))                                      # (draws, rows, dims)
log_ratio = (posterior.log_prob(draws) - prior.log_prob(draws)).sum(-1)    # (draws, rows)
monte_carlo_kl = log_ratio.mean(0)
standard_error = log_ratio.std(0) / np.sqrt(len(draws))

print(f"{'row':>3} {'closed form':>12} {'Monte Carlo':>12} {'diff':>8} {'diff / SE':>9} {'averaged (wrong)':>17}")
for row in range(len(check_means)):
    diff = (closed_form_kl[row] - monte_carlo_kl[row]).item()
    print(f"{row:>3} {closed_form_kl[row]:12.4f} {monte_carlo_kl[row]:12.4f} {diff:8.4f} "
          f"{diff / standard_error[row].item():9.2f} {closed_form_kl[row].item() / LATENT_DIM:17.4f}")
max_z = ((closed_form_kl - monte_carlo_kl).abs() / standard_error).max().item()
print(f"largest |diff| in standard errors: {max_z:.2f}")
results["check_kl"] = {"closed_form": closed_form_kl.tolist(), "monte_carlo": monte_carlo_kl.tolist(),
                       "standard_error": standard_error.tolist()}

row  closed form  Monte Carlo     diff diff / SE  averaged (wrong)
  0      10.6361      10.6268   0.0092      0.88            1.3295
  1      21.9298      21.9343  -0.0045     -0.09            2.7412
  2      20.1421      20.1374   0.0047      0.16            2.5178
  3       9.3738       9.3593   0.0145      1.21            1.1717
  4       2.6726       2.6702   0.0024      0.37            0.3341
  5      16.7566      16.7472   0.0095      0.48            2.0946
largest |diff| in standard errors: 1.21


The closed form matches the Monte Carlo estimate on all six rows, the worst one is 1.21 standard errors off, which is just sampling noise. The last column is what I'd get by averaging over the 8 dimensions instead of summing: every value is exactly one eighth of the real KL, so a model trained on it would be at beta 1/8 without anything looking wrong.

## Check 2: does the reconstruction gradient reach the encoder?

The point of the trick is that the sample is a differentiable function of the mean and std. To test it I take one batch of 64 training images, use only the reconstruction term, and read the gradient norm on the encoder's mean layer. Then I do the same with the sample detached, and with the sampling wrapped in `torch.no_grad`, which are the two silent mistakes. A pass is a positive norm with the trick and exactly 0 for both broken versions, while the decoder still gets a gradient in all three (that's why the loss would still fall).

In [6]:
def gradient_norms(model, xb, sampling, include_kl=False):
    model.zero_grad()
    mean, logvar = model.encode(xb)
    if sampling == "trick":
        z = model.reparameterize(mean, logvar)
    elif sampling == "detached":
        z = model.reparameterize(mean, logvar).detach()
    elif sampling == "no_grad":
        with torch.no_grad():
            z = model.reparameterize(mean, logvar)
    loss = reconstruction_term(model.decode(z), xb).mean()
    if include_kl:
        loss = loss + kl_to_standard_normal(mean, logvar).mean()
    if loss.requires_grad:
        loss.backward()

    def norm(layer):
        return 0.0 if layer.weight.grad is None else layer.weight.grad.norm().item()
    return {"encoder mean": norm(model.enc_mean), "encoder logvar": norm(model.enc_logvar),
            "decoder output": norm(model.dec[-1])}


torch.manual_seed(0)
vae = VAE()
batch = Xtr[:64]
gradient_check = {}
for sampling, include_kl in [("trick", False), ("detached", False), ("no_grad", False), ("detached", True)]:
    torch.manual_seed(1)
    label = f"{sampling}, {'recon + KL' if include_kl else 'recon only'}"
    gradient_check[label] = gradient_norms(vae, batch, sampling, include_kl)
    print(f"{label:<22}", "  ".join(f"{k}: {v:.6f}" for k, v in gradient_check[label].items()))
results["check_gradients"] = gradient_check

trick, recon only      encoder mean: 0.797573  encoder logvar: 0.301622  decoder output: 5.671147
detached, recon only   encoder mean: 0.000000  encoder logvar: 0.000000  decoder output: 5.671147
no_grad, recon only    encoder mean: 0.000000  encoder logvar: 0.000000  decoder output: 5.671147
detached, recon + KL   encoder mean: 0.266565  encoder logvar: 0.262317  decoder output: 5.671147


**With the trick:** the reconstruction term alone sends 0.80 to the encoder's mean layer and 0.30 to the log-variance layer.
#
# **Detached or `no_grad`:** both encoder norms are exactly 0.000000, while the decoder gets the same 5.67 in every case. So the decoder keeps learning and the encoder is cut off from the reconstruction.
#
# **Detached plus KL:** the encoder does get a gradient again (0.27), but all of it comes from the KL, and the KL only pulls the code towards the prior.

To see what the detached mistake does over a whole run, I train one VAE at beta 1 with the sample detached and compare it with the correct one on the same seed. Both losses should fall, so the loss curve alone wouldn't catch it. What should give it away is reconstruction from the posterior mean.

In [7]:
class DetachedSampleVAE(VAE):
    def reparameterize(self, mean, logvar):
        return super().reparameterize(mean, logvar).detach()


@torch.no_grad()
def vae_reconstruction_mse(model, X):
    mean, _ = model.encode(X)
    return ((torch.sigmoid(model.decode(mean)) - X) ** 2).mean().item()


detach_demo = {}
for name, model_class in [("trick", VAE), ("detached", DetachedSampleVAE)]:
    torch.manual_seed(0)
    model = model_class()
    history = S.train(model, make_vae_loss(1.0), Xtr, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, seed=0)
    with torch.no_grad():
        recon, kl = vae_terms(model, Xte)
    detach_demo[name] = {"first_loss": history[0], "last_loss": history[-1], "test_kl": kl.mean().item(),
                         "test_mse_from_mean": vae_reconstruction_mse(model, Xte)}
    print(f"{name:<9} loss {history[0]:.2f} -> {history[-1]:.2f}   test KL {kl.mean():.3f}   "
          f"test MSE from posterior mean {detach_demo[name]['test_mse_from_mean']:.4f}")
print(f"MSE of predicting the mean training image: {((Xtr.mean(0) - Xte) ** 2).mean():.4f}")
results["check_detached_training"] = detach_demo

trick     loss 38.84 -> 23.95   test KL 3.478   test MSE from posterior mean 0.0259


detached  loss 39.65 -> 27.10   test KL 0.000   test MSE from posterior mean 0.0738
MSE of predicting the mean training image: 0.0738


Both losses fall (38.8 to 24.0 with the trick, 39.7 to 27.1 detached), so the training curve alone would look fine. The detached model's KL went to 0.000, which means every posterior is the prior, and its test MSE from the posterior mean is 0.0738, the same as predicting the mean training image for everything. Its decoder learned to output the average digit and nothing else.
#
# The correct VAE only has a KL of 3.5 nats on the test images. That's a lot less information in the code than I expected at beta 1, and I want to keep an eye on it in the comparison with the autoencoder.

## Check 3: the loss at beta 1 is the single-sample negative ELBO

The ELBO for one image is `E_q[log p(x | z)] - KL(q(z | x) || p(z))`, and with one draw of `z` the first part becomes `log p(x | z)` at that draw. I rebuild that with `torch.distributions` (a Bernoulli on the logits and the library's own Normal KL), using the same draw as my loss by resetting the seed, and compare it with my loss at beta 1. A pass is a difference at float precision. I also average the ELBO's sampled form, `log p(x | z) + log p(z) - log q(z | x)`, over 2,000 draws per image, and my loss over the same number of draws. Those two should agree up to Monte Carlo noise.

In [8]:
torch.manual_seed(0)
vae = VAE()
S.train(vae, make_vae_loss(1.0), Xtr, epochs=20, lr=LEARNING_RATE, batch_size=BATCH_SIZE, seed=0)  # a partly trained model
check_batch = Xte[:64]

with torch.no_grad():
    torch.manual_seed(5)
    my_loss = make_vae_loss(1.0)(vae, check_batch, None).item()

    torch.manual_seed(5)
    mean, logvar = vae.encode(check_batch)
    std = torch.exp(0.5 * logvar)
    z = mean + std * torch.randn_like(mean)
    log_likelihood = torch.distributions.Bernoulli(logits=vae.decode(z), validate_args=False).log_prob(check_batch).sum(1)
    library_kl = torch.distributions.kl_divergence(torch.distributions.Normal(mean, std),
                                                   torch.distributions.Normal(0.0, 1.0)).sum(1)
    negative_elbo = -(log_likelihood - library_kl).mean().item()
print(f"my loss at beta 1:        {my_loss:.6f}")
print(f"-ELBO, torch.distributions: {negative_elbo:.6f}   difference {abs(my_loss - negative_elbo):.2e}")

with torch.no_grad():
    n_draws = 2000
    posterior = torch.distributions.Normal(mean, std)
    z_draws = posterior.sample((n_draws,))                                              # (draws, 64, 8)
    log_px = torch.distributions.Bernoulli(logits=vae.decode(z_draws), validate_args=False).log_prob(check_batch).sum(-1)
    sampled_elbo = log_px + prior.log_prob(z_draws).sum(-1) - posterior.log_prob(z_draws).sum(-1)
    per_draw_loss = torch.stack([make_vae_loss(1.0)(vae, check_batch, None) for _ in range(n_draws)])
print(f"over {n_draws} draws: my loss {per_draw_loss.mean():.4f} +/- {per_draw_loss.std() / np.sqrt(n_draws):.4f}   "
      f"-ELBO (sampled form) {-sampled_elbo.mean(1).mean():.4f} +/- {sampled_elbo.mean(1).std() / np.sqrt(n_draws):.4f}")
results["check_elbo"] = {"my_loss": my_loss, "negative_elbo": negative_elbo}

my loss at beta 1:        25.414011
-ELBO, torch.distributions: 25.414011   difference 0.00e+00


over 2000 draws: my loss 25.2211 +/- 0.0046   -ELBO (sampled form) 25.2142 +/- 0.0020


With the same draw my loss and the `torch.distributions` version are identical (difference 0.00e+00). Over 2,000 draws my loss averages 25.221 and the sampled form of the ELBO averages 25.214, 0.007 apart, which is about 1.4 combined standard errors. So at beta 1 the loss is the negative ELBO, and the closed-form KL just removes the noise of sampling it.

## Same bottleneck: autoencoder against VAE

The question here is whether the bottleneck alone makes something I can sample from. I train the starter's `AE` and my VAE at beta 1 with everything matched (8-dimensional code, 64 hidden units, 200 epochs, 2e-3, batch 64) on seeds 0, 1 and 2. The seed sets both the initial weights and the shuffling. For each run I measure:

- **Test reconstruction MSE:** the AE's own output, and for the VAE the decoder applied to the posterior mean.
- **Prior samples:** 359 codes drawn from N(0, I), decoded, and scored with the starter against the test features.
- **Code spread:** the standard deviation of each code dimension over the 1,438 training images.

If the bottleneck were enough, both models would score about the same on prior samples. What I expect instead is that the AE's codes sit wherever training put them, so N(0, I) lands in places its decoder never saw.

In [9]:
def train_vae(beta, seed, model_class=None, y=None):
    torch.manual_seed(seed)
    model = (model_class or VAE)()
    S.train(model, make_vae_loss(beta), Xtr, y=y, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, seed=seed)
    return model.eval()


def train_ae(seed):
    torch.manual_seed(seed)
    model = S.AE(d_hidden=HIDDEN_DIM, d_latent=LATENT_DIM)
    S.train(model, S.ae_loss, Xtr, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, seed=seed)
    return model.eval()


@torch.no_grad()
def ae_prior_samples(model, n, seed):
    z = torch.randn(n, LATENT_DIM, generator=torch.Generator().manual_seed(seed))
    return model.decode(z).clamp(0, 1)


def vae_prior_samples(model, n, seed):
    return model.sample(n, generator=torch.Generator().manual_seed(seed))


def score(samples):
    return S.score_samples(samples, test_features, feature_net, Xtr)


ae_models, vae_models, comparison = {}, {}, {}
for seed in SEEDS:
    ae_models[seed] = train_ae(seed)
    vae_models[seed] = train_vae(1.0, seed)
    with torch.no_grad():
        ae_codes = ae_models[seed].encode(Xtr)
        vae_means, vae_logvars = vae_models[seed].encode(Xtr)
    comparison[seed] = {
        "ae": {"test_mse": S.reconstruction_mse(ae_models[seed], Xte),
               **score(ae_prior_samples(ae_models[seed], N_SAMPLES, seed=100 + seed)),
               "code_std": ae_codes.std(0).tolist(), "code_mean": ae_codes.mean(0).tolist()},
        "vae": {"test_mse": vae_reconstruction_mse(vae_models[seed], Xte),
                **score(vae_prior_samples(vae_models[seed], N_SAMPLES, seed=100 + seed)),
                "code_std": vae_means.std(0).tolist(), "code_mean": vae_means.mean(0).tolist(),
                "posterior_std": torch.exp(0.5 * vae_logvars).mean(0).tolist()},
    }

print(f"{'model':<5} {'seed':>4} {'test MSE':>9} {'Frechet':>8} {'precision':>9} {'recall':>7} {'nearest train':>13}")
for name in ["ae", "vae"]:
    for seed in SEEDS:
        row = comparison[seed][name]
        print(f"{name.upper():<5} {seed:>4} {row['test_mse']:9.4f} {row['frechet']:8.2f} {row['precision']:9.3f} "
              f"{row['recall']:7.3f} {row['nearest_train']:13.3f}")
results["ae_vs_vae"] = comparison

model seed  test MSE  Frechet precision  recall nearest train
AE       0    0.0153    61.89     0.393   0.858         2.004
AE       1    0.0151    95.68     0.354   0.749         2.120
AE       2    0.0149   102.75     0.465   0.694         2.278
VAE      0    0.0259    14.61     0.889   0.496         1.148
VAE      1    0.0265    13.30     0.900   0.496         1.128
VAE      2    0.0264    13.96     0.897   0.437         1.128


**Reconstruction:** the AE wins on every seed, 0.0149 to 0.0153 against 0.0259 to 0.0265 for the VAE. That's expected, the VAE pays for its KL term with a code that carries less detail.
#
# **Prior samples:** the VAE wins by a lot. Its Fréchet distance is 13.3 to 14.6 on all three seeds, while the AE's is 62 to 103 and moves a lot between seeds. Precision is about 0.9 for the VAE and 0.35 to 0.47 for the AE, so most of the AE's samples don't land near any real digit in feature space.
#
# **Recall:** this is the one column the AE wins (0.69 to 0.86 against 0.44 to 0.50). Its samples are so scattered that some of them fall near most of the real digits, which is coverage by noise, not by good samples.

Now the spread of the codes, one column per dimension. For the AE it's the code itself, for the VAE it's the posterior mean, and I also print the VAE's average posterior std (how much noise the encoder adds on top of the mean). If the VAE's codes fill the prior, the mean's spread and the posterior std together should come out near 1 in each dimension.

In [10]:
def format_row(values):
    return " ".join(f"{v:6.2f}" for v in values)


print(" " * 30 + " ".join(f"{'dim ' + str(d):>6}" for d in range(LATENT_DIM)))
for seed in SEEDS:
    print(f"AE   seed {seed}  std of code         {format_row(comparison[seed]['ae']['code_std'])}")
    print(f"AE   seed {seed}  mean of code        {format_row(comparison[seed]['ae']['code_mean'])}")
for seed in SEEDS:
    print(f"VAE  seed {seed}  std of mean         {format_row(comparison[seed]['vae']['code_std'])}")
    print(f"VAE  seed {seed}  avg posterior std   {format_row(comparison[seed]['vae']['posterior_std'])}")
    total_variance = np.array(comparison[seed]['vae']['code_std']) ** 2 + np.array(comparison[seed]['vae']['posterior_std']) ** 2
    print(f"VAE  seed {seed}  sqrt(total var)     {format_row(np.sqrt(total_variance))}")

                               dim 0  dim 1  dim 2  dim 3  dim 4  dim 5  dim 6  dim 7
AE   seed 0  std of code           0.69   0.87   0.57   0.62   0.67   0.82   0.54   0.67
AE   seed 0  mean of code          0.59   0.12   0.53  -0.81   0.14   0.03   0.43  -0.38
AE   seed 1  std of code           0.63   0.57   0.75   0.65   1.09   0.66   0.74   0.65
AE   seed 1  mean of code          0.10   0.85  -0.08   0.13   0.59   0.82  -0.75   0.27
AE   seed 2  std of code           0.72   0.76   0.75   0.71   0.61   0.71   0.77   0.73
AE   seed 2  mean of code          0.01  -0.45   0.33   0.83   0.64  -1.17   0.51   0.08
VAE  seed 0  std of mean           0.94   0.96   0.94   0.48   0.02   0.82   0.02   0.03
VAE  seed 0  avg posterior std     0.41   0.39   0.42   0.88   0.99   0.57   1.00   1.00
VAE  seed 0  sqrt(total var)       1.02   1.04   1.03   1.00   0.99   1.00   1.00   1.00
VAE  seed 1  std of mean           0.90   0.44   0.81   0.91   0.89   0.02   0.02   0.04
VAE  seed 1  avg posteri

**AE codes:** each dimension has a std of 0.54 to 1.09 and a mean that's off zero by up to 1.17. Nothing in the AE's loss mentions N(0, I), so where the codes end up is an accident of the initialisation, which is why the three seeds differ.
#
# **VAE codes:** the total spread (posterior mean plus posterior noise) is 0.99 to 1.04 in every dimension on every seed, which is exactly the prior's 1. The KL term put it there.
#
# **Unused dimensions:** in each VAE run three dimensions have a posterior-mean std of 0.02 to 0.04 and a posterior std of 1.00. Those dimensions ignore the input and just pass on prior noise, so at beta 1 the VAE is only really using 5 of its 8 dimensions. I'll count these properly in the beta sweep.
#
# The AE's spreads aren't wildly off 1 though, so the per-dimension numbers alone don't fully explain a Fréchet distance of 60 to 100.

I also want to look at the samples, not just the scores. The figure has a row of test images and their reconstructions for both models, then prior samples from both, all from seed 0.

In [11]:
def save_rows(rows, path, title=None):
    """rows: list of (label, images N by 64). One labelled row per entry."""
    ncols = max(len(images) for _, images in rows)
    fig, axes = plt.subplots(len(rows), ncols, figsize=(ncols * 0.75 + 1.6, len(rows) * 0.8 + 0.3), squeeze=False)
    for r, (label, images) in enumerate(rows):
        for c in range(ncols):
            ax = axes[r, c]
            ax.set_xticks([]); ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)
            if c < len(images):
                ax.imshow(images[c].detach().clamp(0, 1).reshape(8, 8), cmap="gray_r", vmin=0, vmax=1)
        axes[r, 0].set_ylabel(label, rotation=0, ha="right", va="center", fontsize=8)
    if title:
        fig.suptitle(title, fontsize=10)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


with torch.no_grad():
    shown = Xte[:12]
    vae_means_shown, _ = vae_models[0].encode(shown)
    save_rows([("test images", shown),
               ("AE rebuilt", ae_models[0](shown)),
               ("VAE rebuilt", torch.sigmoid(vae_models[0].decode(vae_means_shown))),
               ("AE from N(0, I)", ae_prior_samples(ae_models[0], 12, seed=100)),
               ("VAE from N(0, I)", vae_prior_samples(vae_models[0], 12, seed=100))],
              "figures/ae_vs_vae_samples.png", title="Autoencoder against VAE at beta 1, seed 0")
print("saved figures/ae_vs_vae_samples.png")

saved figures/ae_vs_vae_samples.png


Both reconstruct recognisable digits, the AE's are a bit sharper. The AE's prior samples are mostly broken strokes with very dark and very light pixels side by side, while the VAE's prior samples look like slightly blurry digits (0, 2, 4, 6, 9 are easy to read).

The per-dimension spread on its own doesn't settle it, so I measure what the decoder actually gets fed. For each model I draw 359 codes from N(0, I) and find each one's distance to the nearest code the decoder was trained on (the AE's training codes, or one posterior draw per training image for the VAE). As the reference I do the same for the 359 test images' codes, which the decoder handles fine. I also print the largest correlation between two code dimensions, since the prior assumes they're independent.

In [12]:
@torch.no_grad()
def code_coverage(model, kind, seed):
    if kind == "ae":
        train_codes, test_codes = model.encode(Xtr), model.encode(Xte)
    else:
        torch.manual_seed(seed)
        train_codes = model.reparameterize(*model.encode(Xtr))
        test_codes = model.reparameterize(*model.encode(Xte))
    prior_codes = torch.randn(N_SAMPLES, LATENT_DIM, generator=torch.Generator().manual_seed(100 + seed))
    prior_gap = torch.cdist(prior_codes, train_codes).min(1).values.mean().item()
    test_gap = torch.cdist(test_codes, train_codes).min(1).values.mean().item()
    correlation = torch.corrcoef(train_codes.T) - torch.eye(LATENT_DIM)
    return {"prior_to_train": prior_gap, "test_to_train": test_gap, "ratio": prior_gap / test_gap,
            "max_abs_correlation": correlation.abs().max().item()}


print(f"{'model':<5} {'seed':>4} {'N(0,I) -> train':>16} {'test -> train':>14} {'ratio':>6} {'max |corr|':>11}")
for name, models in [("ae", ae_models), ("vae", vae_models)]:
    for seed in SEEDS:
        coverage = code_coverage(models[seed], name, seed)
        comparison[seed][name]["coverage"] = coverage
        print(f"{name.upper():<5} {seed:>4} {coverage['prior_to_train']:16.3f} {coverage['test_to_train']:14.3f} "
              f"{coverage['ratio']:6.2f} {coverage['max_abs_correlation']:11.2f}")

model seed  N(0,I) -> train  test -> train  ratio  max |corr|
AE       0            1.945          0.514   3.78        0.48
AE       1            2.060          0.530   3.88        0.50
AE       2            2.104          0.537   3.92        0.45
VAE      0            1.346          1.340   1.00        0.07
VAE      1            1.310          1.301   1.01        0.07
VAE      2            1.310          1.308   1.00        0.05


This is what explains it. A draw from N(0, I) is on average 1.9 to 2.1 away from the nearest AE training code, about 3.8 to 3.9 times further than a test image's code is. So the AE's codes fill only a thin, correlated (up to 0.50) part of the space and most prior draws land in the gaps between them, where the decoder was never trained. For the VAE the two distances are the same (ratio 1.00 to 1.01) and the dimensions are close to uncorrelated (at most 0.07). A prior draw looks to its decoder like any other code it trained on.

So the bottleneck forces both models to learn structure, but only the VAE trained its decoder on the region the prior covers. That's why one can generate and the other can't.

## Sweep the KL weight

My prediction is in the README, written before this cell ran. I train VAEs at beta 0.1, 0.5, 1, 2 and 4 on seeds 0, 1 and 2, and only beta changes between runs. Beta 1 reuses the three models from the comparison above, which were trained under exactly these settings. For each run:

- **Reconstruction and KL:** both terms averaged over the 359 test images, with one posterior draw per image (fixed seed).
- **Active dimensions:** a dimension counts if its posterior mean has a variance above 0.01 across the 1,438 training images.
- **Sample scores:** Fréchet distance, precision and recall of 359 prior samples against the test features, with the same feature network as everywhere else.

A collapsed run is one with 0 active dimensions, where the decoder can only ever see prior noise.

In [13]:
BETAS = [0.1, 0.5, 1.0, 2.0, 4.0]


@torch.no_grad()
def active_dimensions(model, threshold=0.01):
    mean, _ = model.encode(Xtr)
    return int((mean.var(0) > threshold).sum().item())


@torch.no_grad()
def evaluate_vae(model, seed):
    torch.manual_seed(1000 + seed)
    recon, kl = vae_terms(model, Xte)
    return {"reconstruction": recon.mean().item(), "kl": kl.mean().item(),
            "active": active_dimensions(model),
            **score(vae_prior_samples(model, N_SAMPLES, seed=100 + seed))}


sweep_start = time.perf_counter()
sweep_models, sweep = {}, {}
for beta in BETAS:
    for seed in SEEDS:
        sweep_models[beta, seed] = vae_models[seed] if beta == 1.0 else train_vae(beta, seed)
        sweep[beta, seed] = evaluate_vae(sweep_models[beta, seed], seed)
print(f"sweep took {time.perf_counter() - sweep_start:.0f}s")

print(f"{'beta':>4} {'seed':>4} {'recon':>7} {'KL':>6} {'active':>6} {'Frechet':>8} {'precision':>9} {'recall':>7}")
for beta in BETAS:
    for seed in SEEDS:
        row = sweep[beta, seed]
        print(f"{beta:>4} {seed:>4} {row['reconstruction']:7.2f} {row['kl']:6.2f} {row['active']:>6} "
              f"{row['frechet']:8.2f} {row['precision']:9.3f} {row['recall']:7.3f}")
results["beta_sweep"] = {f"{beta}_{seed}": row for (beta, seed), row in sweep.items()}

sweep took 99s
beta seed   recon     KL active  Frechet precision  recall
 0.1    0   17.28  14.20      8    11.17     0.660   0.852
 0.1    1   17.06  14.20      8    12.77     0.719   0.869
 0.1    2   17.15  14.27      8    13.32     0.769   0.808
 0.5    0   19.25   5.83      7     9.58     0.822   0.741
 0.5    1   19.26   5.96      7     8.40     0.841   0.747
 0.5    2   19.11   6.15      7    10.50     0.825   0.769
 1.0    0   20.88   3.48      5    14.61     0.889   0.496
 1.0    1   21.20   3.22      5    13.30     0.900   0.496
 1.0    2   21.13   3.28      5    13.96     0.897   0.437
 2.0    0   24.19   1.26      3    43.91     0.936   0.064
 2.0    1   24.16   1.16      3    46.62     0.905   0.092
 2.0    2   24.22   1.26      4    43.35     0.928   0.100
 4.0    0   27.26   0.00      0   218.99     1.000   0.000
 4.0    1   27.25   0.00      0   219.64     1.000   0.000
 4.0    2   27.26   0.00      0   224.31     1.000   0.000


**The trade:** going from beta 0.1 to 4, reconstruction gets worse every step (17.2 to 27.3) and the KL drops (14.2 to 0.00). The three seeds land within about 0.3 of each other on both terms at every beta.

**Active dimensions:** 8, 7, 5, 3 (4 on seed 2) and 0. My prediction had 8 or 7, about 6, 5, 2 or 3, and 0 or 1, so it was close at every point except 0.5, which kept one more dimension than I guessed.

**Collapse:** all three seeds at beta 4 have a KL of 0.00 and no active dimensions, and their reconstruction term, 27.26, is the same as the detached-sample model from check 2. I predicted collapse at 4 on at least one seed, and it's all three.

**Best samples:** beta 0.5 has the lowest Fréchet distance on every seed (8.4 to 10.5), not beta 1 as I predicted. Beta 0.1 is second on every seed (11.2 to 13.3) and beta 1 is only third (13.3 to 14.6). Precision is lowest at 0.1 (0.66 to 0.77), so more prior draws do land off the digits like I expected, but its much higher recall makes up for that in the Fréchet distance. I overestimated how much the samples lose by having a weak KL, and underestimated what beta 1 loses by switching off three dimensions. Recall falls fastest of the three scores as beta rises: 0.85 at 0.1, about 0.75 at 0.5, 0.44 to 0.50 at 1 and below 0.10 at 2.

**Inconvenient result:** the collapsed models get a precision of 1.000. Every sample is the same average digit, which sits close enough to the real test features to count as "inside" their neighbourhoods. The Fréchet distance (219 to 224) and recall (0.000) catch it, precision rewards it.

Beta 2 still has 3 or 4 active dimensions and beta 4 has none, so the collapse is somewhere in between. To find it I add 2.5, 3 and 3.5 under the same settings and seeds. These are extra runs on top of the five weights the handout asks for.

In [14]:
FINE_BETAS = [2.5, 3.0, 3.5]
for beta in FINE_BETAS:
    for seed in SEEDS:
        sweep_models[beta, seed] = train_vae(beta, seed)
        sweep[beta, seed] = evaluate_vae(sweep_models[beta, seed], seed)

print(f"{'beta':>4} {'seed':>4} {'recon':>7} {'KL':>6} {'active':>6} {'Frechet':>8} {'precision':>9} {'recall':>7}")
for beta in [2.0] + FINE_BETAS + [4.0]:
    for seed in SEEDS:
        row = sweep[beta, seed]
        print(f"{beta:>4} {seed:>4} {row['reconstruction']:7.2f} {row['kl']:6.2f} {row['active']:>6} "
              f"{row['frechet']:8.2f} {row['precision']:9.3f} {row['recall']:7.3f}")
results["beta_sweep"] = {f"{beta}_{seed}": row for (beta, seed), row in sweep.items()}

beta seed   recon     KL active  Frechet precision  recall
 2.0    0   24.19   1.26      3    43.91     0.936   0.064
 2.0    1   24.16   1.16      3    46.62     0.905   0.092
 2.0    2   24.22   1.26      4    43.35     0.928   0.100
 2.5    0   25.44   0.66      3    68.06     0.919   0.053
 2.5    1   25.42   0.60      3    75.53     0.933   0.014
 2.5    2   25.66   0.62      3    75.36     0.939   0.014
 3.0    0   27.09   0.07      2   157.78     0.989   0.000
 3.0    1   26.93   0.07      1   164.50     0.992   0.000
 3.0    2   27.08   0.08      2   163.56     0.981   0.000
 3.5    0   27.26   0.00      0   211.41     1.000   0.000
 3.5    1   27.24   0.00      0   216.69     1.000   0.000
 3.5    2   27.26   0.00      0   220.17     1.000   0.000
 4.0    0   27.26   0.00      0   218.99     1.000   0.000
 4.0    1   27.25   0.00      0   219.64     1.000   0.000
 4.0    2   27.26   0.00      0   224.31     1.000   0.000


**Where it collapses:** between 3 and 3.5. At beta 3 the KL is already down to 0.07 to 0.08 nats with 1 or 2 dimensions barely above the threshold, and the Fréchet distance is 158 to 165, so it's nearly gone. At 3.5 all three seeds have a KL of 0.00, the same as at 4. The drop isn't gradual at the end either: from 2.5 to 3 the KL falls from about 0.6 to 0.07.

Precision climbs towards 1.000 the closer a model gets to collapse (0.92 to 0.94 at 2.5, 0.98 to 0.99 at 3), which is the opposite of what a quality measure should do here.

## Look at what a collapsed model produces

For the beta 4 runs I want to see the samples and check that the decoder really ignores its input. If it does, every prior draw decodes to the same image, so the pixel std across 359 samples should be about 0, and that image should be the mean training image.

In [15]:
with torch.no_grad():
    for seed in SEEDS:
        collapsed_samples = vae_prior_samples(sweep_models[4.0, seed], N_SAMPLES, seed=100 + seed)
        pixel_spread = collapsed_samples.std(0).max().item()
        gap_to_mean_image = (collapsed_samples.mean(0) - Xtr.mean(0)).abs().max().item()
        posterior_means, posterior_logvars = sweep_models[4.0, seed].encode(Xtr)
        print(f"seed {seed}: largest pixel std across samples {pixel_spread:.5f}   "
              f"largest gap to the mean training image {gap_to_mean_image:.4f}   "
              f"posterior means within {posterior_means.abs().max():.4f} of 0, "
              f"posterior std {torch.exp(0.5 * posterior_logvars).min():.3f} to {torch.exp(0.5 * posterior_logvars).max():.3f}")

seed 0: largest pixel std across samples 0.01568   largest gap to the mean training image 0.0113   posterior means within 0.0797 of 0, posterior std 0.984 to 1.007
seed 1: largest pixel std across samples 0.01384   largest gap to the mean training image 0.0092   posterior means within 0.1130 of 0, posterior std 0.986 to 1.008
seed 2: largest pixel std across samples 0.01187   largest gap to the mean training image 0.0107   posterior means within 0.0450 of 0, posterior std 0.988 to 1.005


On every seed the 359 samples differ by at most 0.016 in any pixel, and their average is within 0.011 of the mean training image. Every posterior is N(0, 1) to within about 0.1 in the mean and 0.02 in the std, whatever image goes in. So the encoder throws the input away, the decoder ignores the code, and the model has fallen back on the cheapest thing that still lowers the reconstruction term: the average digit.

The figure has 12 prior samples per beta from seed 0, and the mean training image at the bottom for comparison.

In [16]:
rows = [(f"beta {beta:g} ({sweep[beta, 0]['active']} active)", vae_prior_samples(sweep_models[beta, 0], 12, seed=100))
        for beta in BETAS + FINE_BETAS if beta != 4.0]
rows.sort(key=lambda row: float(row[0].split()[1]))
rows.append((f"beta 4 ({sweep[4.0, 0]['active']} active)", vae_prior_samples(sweep_models[4.0, 0], 12, seed=100)))
rows.append(("mean train image", Xtr.mean(0, keepdim=True)))
save_rows(rows, "figures/beta_sweep_samples.png", title="VAE prior samples by KL weight, seed 0")
print("saved figures/beta_sweep_samples.png")

saved figures/beta_sweep_samples.png


Going down the rows the digits get thicker and blurrier, and from beta 3 most samples are the same grey blob as the mean image. At beta 2 and 2.5 there are still clear 1s, 0s and 6s, but the 3 to 4 dimensions left only seem to cover a few digit shapes, which fits recall dropping below 0.10. Beta 0.1 has the sharpest strokes but also the odd broken sample (the fourth one), which is the lower precision showing up.